In [1]:
import warnings
from langchain_core._api.deprecation import LangChainDeprecationWarning
warnings.filterwarnings("ignore", category=LangChainDeprecationWarning)

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_community.vectorstores import FAISS
from transformers import pipeline
import torch

C:\Users\mohan\AppData\Local\Temp\ipykernel_15264\1172610142.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [2]:
loader = TextLoader("C:/Users/mohan/Downloads/# Data Science Scope in IT, Eligibi.txt", encoding="utf-8")
docs = loader.load()

print(f"Loaded {len(docs)} document(s)")

Loaded 1 document(s)


In [3]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(docs)

print(f"Created {len(chunks)} chunk(s)")
for i, chunk in enumerate(chunks):
    print(f"Chunk {i}:")
    print(chunk.page_content)
    print("-" * 30)

Created 81 chunk(s)
Chunk 0:
# Data Science: Scope in IT, Eligibility, and Requirements for a Data Scientist

## Introduction
------------------------------
Chunk 1:
Data Science is one of the most important and rapidly growing fields in today's digital world. It is an interdisciplinary domain that combines computer science, mathematics, statistics, machine
------------------------------
Chunk 2:
science, mathematics, statistics, machine learning, and domain knowledge to extract meaningful insights from large volumes of structured and unstructured data. Organizations across industries
------------------------------
Chunk 3:
data. Organizations across industries generate enormous amounts of data every day through websites, mobile applications, social media, sensors, financial transactions, and business operations. Data
------------------------------
Chunk 4:
transactions, and business operations. Data Science helps transform this raw data into valuable information that supports better d

In [4]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
embeddings

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [5]:
sample_vector = embeddings.embed_query("What is Data Science?")
print(f"Embedding length: {len(sample_vector)}")
print(sample_vector[:10], "...")

Embedding length: 384
[-0.03661505877971649, 0.019506828859448433, -0.051343001425266266, 0.0335746705532074, -0.018491152673959732, -0.13787367939949036, 6.481021409854293e-05, 0.0009964104974642396, -0.033022843301296234, 0.0681999996304512] ...


In [6]:
db = FAISS.from_documents(chunks, embeddings)

stored_docs = db.docstore._dict
for i, (doc_id, doc) in enumerate(stored_docs.items()):
    print(f"Document {i+1}")
    print("Content:")
    print(doc.page_content)
    print("-" * 60)

Document 1
Content:
# Data Science: Scope in IT, Eligibility, and Requirements for a Data Scientist

## Introduction
------------------------------------------------------------
Document 2
Content:
Data Science is one of the most important and rapidly growing fields in today's digital world. It is an interdisciplinary domain that combines computer science, mathematics, statistics, machine
------------------------------------------------------------
Document 3
Content:
science, mathematics, statistics, machine learning, and domain knowledge to extract meaningful insights from large volumes of structured and unstructured data. Organizations across industries
------------------------------------------------------------
Document 4
Content:
data. Organizations across industries generate enormous amounts of data every day through websites, mobile applications, social media, sensors, financial transactions, and business operations. Data
--------------------------------------------------------

In [7]:
retriever = db.as_retriever(
    search_kwargs={"k": 5})

In [8]:
pipe = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-0.5B-Instruct",
    torch_dtype=torch.float32,     
    device=0 if torch.cuda.is_available() else -1,
    max_new_tokens=100,            
    temperature=0.1,
    do_sample=False,
    top_p=0.9,
    return_full_text=False
)

llm = HuggingFacePipeline(pipeline=pipe)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'temperature', 'top_p', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [9]:
NOT_IN_TEXT = "This is not given in the text."
RELEVANCE_THRESHOLD = 1.1 
SYSTEM_PROMPT = (
    "You are a strict retrieval-augmented assistant. Answer ONLY using the given context. "
    "Never use outside knowledge. If the answer is not fully contained in the context, "
    f"reply with exactly: '{NOT_IN_TEXT}'. Keep answers to 2-3 lines."
)

def generate(messages) -> str:
    out = pipe(messages)[0]["generated_text"]
    if isinstance(out, list):
        return out[-1]["content"].strip()
    return str(out).strip()

def self_rag_part(question: str, k: int = 5) -> str:
    results = db.similarity_search_with_score(question, k=k)
    relevant = [doc for doc, score in results if score <= RELEVANCE_THRESHOLD]

    if not relevant:
        return NOT_IN_TEXT

    context = "\n\n".join(doc.page_content for doc in relevant)
    answer = generate([
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"}
    ])

    if NOT_IN_TEXT.lower() in answer.lower() or "don't have enough information" in answer.lower():
        return NOT_IN_TEXT
    critique = generate([
        {"role": "system", "content": "Reply with ONLY the single word YES or NO."},
        {"role": "user", "content": (
            f"Context:\n{context}\n\nQuestion: {question}\n\nAnswer: {answer}\n\n"
            "Is the answer fully and only supported by the context?"
        )}
    ]).upper()

    if "NO" in critique:
        return NOT_IN_TEXT\

    return f"Context:\n{context}\n\nQuestion:\n{question}\n\nAnswer:\n{answer}"

In [10]:
response = self_rag_part("what are the educational qualifications for datascience?")
print(response)

[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the docum

Context:
Professionals can specialize further based on their interests and expertise.

## Who is Eligible for Data Science?

## Who is Eligible for Data Science?

Students and professionals from various educational backgrounds can pursue Data Science if they possess strong analytical and technical skills.

* Master's degree in Data Science, Artificial Intelligence, Business Analytics, or Computer Science (optional but beneficial).
* Relevant certifications from recognized learning platforms.

# Data Science: Scope in IT, Eligibility, and Requirements for a Data Scientist

## Introduction

## Soft Skills Required

Apart from technical expertise, Data Scientists should possess:

Question:
what are the educational qualifications for datascience?

Answer:
Students and professionals from various educational backgrounds can pursue Data Science if they possess strong analytical and technical skills. This includes:

1. **Master's degree** in Data Science, Artificial Intelligence, Business Anal